# 03 · EDA: collected data and cleaning

EDA steps 1–2 from the README. What did we collect (volume by game, language and time, coverage against Steam's own totals), and what does cleaning remove?

Data is loaded with `src.db.read`, and all analysis is pandas. Fill the database first with `01_collect.ipynb` and `02_parse_clean.ipynb`.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.ticker import EngFormatter, PercentFormatter

from src.db import read
from src.config import CONFIG

pd.set_option("display.max_colwidth", 120)
# fixed color per language (identity follows the entity, not the rank)
LANG_COLORS = {"english": "#2a78d6", "russian": "#eb6834", "schinese": "#1baf7a", "tchinese": "#eda100"}
plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
    "grid.color": "#e5e5e5", "grid.linewidth": 0.8, "lines.linewidth": 2, "figure.figsize": (11, 4),
})
SERIES_COLOR = "#2a78d6"
PATCH_COLOR = "#a0a0a0"
# charts start at since_date if set, otherwise at each game's first data
SINCE = pd.Timestamp(CONFIG.reviews.since_date or "2000-01-01")

In [ ]:
# With the full collection, pass appids=[...] / languages=[...] to load less.
games = pd.Series({g.appid: g.name for g in CONFIG.games})
reviews = read.reviews()
news = read.news()
players = read.player_counts()

reviews["game"] = reviews["appid"].map(games)
reviews["text_len"] = reviews["review"].fillna("").str.len()
pd.Series({"games": len(games), "reviews": len(reviews), "news": len(news),
           "patches": news["is_patch"].sum(), "player count points": len(players)})

## 1. Collection

### 1.1 Reviews per game × language, and coverage against Steam's totals

`steam_total` is Steam's all-time count for that language. Coverage is below 100% when a game has more reviews than `reviews_per_lang` (only the newest are collected) or when `since_date` is set.

In [ ]:
ours = reviews.groupby(["game", "appid", "language"]).agg(
    collected=("recommendation_id", "size"),
    first_review=("created_at", "min"),
    last_review=("created_at", "max"),
    share_positive=("voted_up", "mean"),
)
steam_total = (
    read.review_totals()
    .sort_values("fetched_at")
    .groupby(["appid", "language"])["total_reviews"]
    .last()
    .rename("steam_total")
)
coverage = ours.join(steam_total, on=["appid", "language"])
coverage["coverage"] = coverage["collected"] / coverage["steam_total"]
coverage.round({"share_positive": 3, "coverage": 3})

In [ ]:
per_game = reviews.groupby(["game", "language"]).size().unstack(fill_value=0)
per_game = per_game[[c for c in CONFIG.reviews.languages if c in per_game]]  # drop odd labels
per_game = per_game.loc[per_game.sum(axis=1).sort_values().index]

ax = per_game.plot.barh(
    stacked=True, color=[LANG_COLORS.get(c, "#888888") for c in per_game.columns],
    edgecolor="white", linewidth=2, width=0.7, figsize=(10, 0.5 * len(per_game) + 1),
)
ax.set(title="Collected reviews per game", xlabel="reviews", ylabel="")
ax.set_axisbelow(True)
ax.grid(axis="y", visible=False)
ax.legend(title="", frameon=False, loc="lower right");

### 1.2 Volume over time

In [ ]:
month = reviews["created_at"].dt.tz_convert(None).dt.to_period("M").dt.to_timestamp()
monthly = reviews.groupby([month, "language"]).size().unstack(fill_value=0)

ax = monthly.plot(color=[LANG_COLORS.get(c, "#888888") for c in monthly.columns])
ax.set(title="Reviews per month by language (all games)", xlabel="", ylabel="reviews")
ax.legend(title="", frameon=False);

### 1.3 Text length and author fields

In [ ]:
reviews.groupby("language")["text_len"].describe(percentiles=[0.1, 0.5, 0.9]).round(0)

In [ ]:
# num_games_owned = 0 almost always means a private profile, not an empty library.
reviews.groupby("language").agg(
    share_hidden_library=("author_num_games_owned", lambda s: (s == 0).mean()),
    median_hours_at_review=("author_playtime_at_review", lambda s: s.median() / 60),
    share_free=("received_for_free", "mean"),
    share_steam_purchase=("steam_purchase", "mean"),
).round(3)

### 1.4 Steam language vs language of the text

`language` is the reviewer's Steam client language, which stands for the audience. `text_language` is what the text is actually written in: the detected language when the detector is sure, otherwise the Steam language. Audience comparisons use `language`; anything that reads the text uses `text_language`.

In [ ]:
pd.crosstab(reviews["language"], reviews["text_language"], normalize="index").round(3)

### 1.5 News and online player data

In [ ]:
pd.DataFrame({
    "news": news.groupby("appid").size(),
    "patches": news.groupby("appid")["is_patch"].sum(),
    "first_news": news.groupby("appid")["published_at"].min().dt.date,
    "player_points": players.groupby("appid").size(),
    "players_from": players.groupby("appid")["ts"].min().dt.date,
    "players_to": players.groupby("appid")["ts"].max().dt.date,
}).rename(index=games)

## 2. Games over time, with patches

One panel per game, because the scales differ too much for one shared axis. Grey vertical lines are news items flagged as patches or updates (`is_patch`). Each panel starts where that game's data starts (or at `since_date` from `configs/config.yaml`, if set).

**Until the full collection:** the per-language cap keeps only the newest reviews, so early weeks look emptier than they really are.

In [ ]:
def to_naive(ts):
    return ts.dt.tz_convert(None)

patches = news[news["is_patch"].astype(bool)].assign(day=lambda d: to_naive(d["published_at"]).dt.floor("D"))
patch_days = patches.groupby("appid")["day"].unique()


def per_game_panels(series: pd.Series, title: str, ylabel: str, ylim=None, percent=False):
    """series has a MultiIndex (appid, date): one panel per game with its patch dates."""
    appids = [a for a in games.index if a in series.index.get_level_values(0)]
    if not appids:
        print(f"{title}: no data yet")
        return
    rows = (len(appids) + 1) // 2
    fig, axes = plt.subplots(rows, 2, figsize=(12, 2.4 * rows), squeeze=False)
    for ax, appid in zip(axes.flat, appids):
        s = series.loc[appid]
        start, end = s.index.min(), s.index.max()
        for day in patch_days.get(appid, []):
            if start <= day <= end:
                ax.axvline(day, color=PATCH_COLOR, linewidth=0.8, alpha=0.5, zorder=0)
        ax.plot(s.index, s.values, color=SERIES_COLOR)  # NaN = no data: the line breaks
        ax.set_title(games[appid], loc="left", fontsize=10)
        ax.set_ylabel(ylabel)
        ax.yaxis.set_major_formatter(PercentFormatter(1.0) if percent else EngFormatter(sep=""))
        locator = mdates.AutoDateLocator()
        ax.xaxis.set_major_locator(locator)
        ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))
        if ylim:
            ax.set_ylim(*ylim)
    for ax in axes.flat[len(appids):]:
        ax.set_visible(False)
    patch_handle = Line2D([], [], color=PATCH_COLOR, linewidth=0.8, label="patch / update")
    fig.legend(handles=[patch_handle], loc="upper right", frameon=False)
    fig.suptitle(title, x=0.01, ha="left")
    fig.tight_layout()


def weekly_with_gaps(s: pd.Series) -> pd.Series:
    """(appid, week) series -> NaN for missing weeks, so lines break instead of bridging gaps."""
    return s.groupby(level=0, group_keys=True).apply(lambda g: g.droplevel(0).asfreq("W-MON"))

### 2.1 Online players (SteamCharts)

Daily average. SteamCharts has monthly averages for older history and 10-minute points for recent months; zeros are tracker outages and are left out.

In [ ]:
recent = players[(to_naive(players["ts"]) >= SINCE) & (players["players"] > 0)]
daily_players = (
    recent.assign(day=to_naive(recent["ts"]).dt.floor("D"))
    .groupby(["appid", "day"])["players"]
    .mean()
)
per_game_panels(daily_players, "Online players (daily average)", "players")

### 2.2 Reviews per week

In [ ]:
week = to_naive(reviews["created_at"]).dt.to_period("W").dt.start_time
recent_reviews = reviews[week >= SINCE]
recent_week = week[week >= SINCE]
weekly = recent_reviews.groupby(["appid", recent_week]).agg(n=("voted_up", "size"), positive=("voted_up", "mean"))
per_game_panels(weekly_with_gaps(weekly["n"]), "Reviews per week (all languages)", "reviews")

### 2.3 Share of positive reviews per week

Only weeks with at least 20 reviews, so single reviews don't swing the line.

In [ ]:
per_game_panels(
    weekly_with_gaps(weekly["positive"].where(weekly["n"] >= 20)),
    "Share of positive reviews per week (all languages)", "positive", ylim=(0, 1), percent=True,
)

## 3. Cleaning

Rules are in `src/clean/rules.py`, and their thresholds are in `configs/config.yaml` → `cleaning:`. `is_empty`, `is_junk` and `is_duplicate_text` set `keep = False`. `is_lang_mismatch` and `is_edited` only flag a review.

### 3.1 How much each rule removes (latest run)

In [ ]:
report = read.cleaning_report()
latest = report[report["run_id"] == report["run_id"].max()]
counts = latest.groupby(["rule", "language"])[["n_flagged", "n_total"]].sum()
(counts["n_flagged"] / counts["n_total"]).unstack("language").style.format("{:.1%}")

### 3.2 Examples per rule

Check by eye that each rule catches what it should.

In [ ]:
def examples(rule: str, n: int = 8) -> pd.DataFrame:
    hits = reviews[reviews[rule].eq(True)]
    return (
        hits.sample(min(n, len(hits)), random_state=0)[["language", "lang_detected", "lang_conf", "review"]]
        .assign(review=lambda d: d["review"].str.replace(r"\s+", " ", regex=True).str.slice(0, 120))
    )

for rule in ["is_empty", "is_junk", "is_duplicate_text", "is_lang_mismatch"]:
    print(f"\n=== {rule}")
    display(examples(rule))

### 3.3 Edited reviews: which date to use

Steam returns `timestamp_created` and `timestamp_updated`. An edit can come months later, often after a patch.

In [ ]:
days_to_edit = (reviews["updated_at"] - reviews["created_at"]).dt.total_seconds() / 86400
pd.DataFrame({
    "share_edited": reviews.groupby("language")["is_edited"].mean(),
    "median_days_to_edit": days_to_edit[reviews["is_edited"].eq(True)].groupby(reviews["language"]).median(),
}).round(3)

**Decision (current default):** use `created_at` as the review date.

- The first opinion is tied to the moment it was written. Moving it to the edit date would bunch old reviews into the edit date.
- The thumbs-up/down vote can flip on edit, and we only see the final state. Reviews with `is_edited` can be excluded from the event analysis (EDA step 6) as a robustness check.

Revisit this if the share of edited reviews is large for a specific game.

### 3.4 Result

In [ ]:
reviews.groupby("language")["keep"].agg(total="size", kept="sum", share_kept="mean").round(3)